# Transform Circuits Data

1. Read bronze `circuits` table
1. Keep only the columns required for analytics (Drop `url` column)
1. Standardise column names using snake_case (`circuitId` → `circuit_id`, `circuitName` → `circuit_name`)
1. Rename columns to make them more meaningful (`lat` → `latitude`, `long` → `longitude`)
1. Filter out rows where `circuit_id` is null (business key validation)
1. Remove duplicate records
1. Transform values of columns `circuit_name` and `locality` to Title Case
1. Write the transformed data to silver `circuits` table


#### Entity Relationship Diagram - Formula1 Schema

![Formula1 Raw Data.png](../../z-course-images/formula1-raw-data-erd.png "Formula1 Raw Data.png")

In [0]:
dbutils.widgets.text("p_batch_id","")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/03.silver-helpers

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.circuits"
silver_table = f"{catalog_name}.{silver_schema}.circuits"

#### Step 1 - Read bronze `circuits` table

In [0]:
# circuits_df = spark.read.option('versionAsOf', 0).table(bronze_table)

In [0]:
from pyspark.sql import functions as F

In [0]:
circuits_df = spark.table(bronze_table).filter(F.col("batch_id") == v_batch_id)

In [0]:
# display(circuits_df)

#### Step 2 - Keep only the columns required for analytics (Drop url column)

In [0]:
# circuits_selected_df = circuits_df.select(
#     "circuitId",
#     "circuitName",
#     "lat",
#     "long",
#     "locality",
#     "country",
#     "ingestion_timestamp",
#     "source_file"
# )

In [0]:
from pyspark.sql import functions as F

In [0]:
circuits_selected_df = circuits_df.select(
    F.col("circuitId"),
    F.col("circuitName"),
    F.col("lat"),
    F.col("long"),
    F.col("locality"),
    F.col("country"),
    F.col("ingestion_timestamp"),
    F.col("source_file"),
    F.col("batch_id")
)

#### Step 3 & 4 - Standardise Column Names
- Standardise column names using snake_case (`circuitId` → `circuit_id`, `circuitName` → `circuit_name`)
- Rename columns to make them more meaningful (`lat` → `latitude`, `long` → `longitude`)

In [0]:
# circuits_renamed_df = (
#     circuits_selected_df
#         .withColumnRenamed("circuitId", "circuit_id")
#         .withColumnRenamed("circuitName", "circuit_name")
#         .withColumnRenamed("lat", "latitude")
#         .withColumnRenamed("long", "longitude")
# )

In [0]:
circuits_renamed_df = (
    circuits_selected_df
        .withColumnsRenamed({
            "circuitId": "circuit_id",
            "circuitName": "circuit_name",
            "lat": "latitude",
            "long": "longitude"
        })
)

In [0]:
display(circuits_renamed_df)

circuit_id,circuit_name,latitude,longitude,locality,country,ingestion_timestamp,source_file,batch_id
null,circuit gilles villeneuve,45.5,-73.5228,montreal,Canada,2026-11-08T09:46:26.697Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/circuits.csv,2025-01
null,losail international circuit,25.49,51.4542,lusail,Qatar,2026-11-08T09:46:26.697Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/circuits.csv,2025-01
adelaide,adelaide street circuit,-34.9272,138.617,adelaide,Australia,2026-11-08T09:46:26.697Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/circuits.csv,2025-01
ain-diab,ain diab,33.5786,-7.6875,casablanca,Morocco,2026-11-08T09:46:26.697Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/circuits.csv,2025-01
aintree,aintree,53.4769,-2.94056,liverpool,UK,2026-11-08T09:46:26.697Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/circuits.csv,2025-01
albert_park,albert park grand prix circuit,-37.8497,144.968,melbourne,Australia,2026-11-08T09:46:26.697Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/circuits.csv,2025-01
americas,circuit of the americas,30.1328,-97.6411,austin,USA,2026-11-08T09:46:26.697Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/circuits.csv,2025-01
anderstorp,scandinavian raceway,57.2653,13.6042,anderstorp,Sweden,2026-11-08T09:46:26.697Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/circuits.csv,2025-01
avus,avus,52.4806,13.2514,berlin,Germany,2026-11-08T09:46:26.697Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/circuits.csv,2025-01
bahrain,bahrain international circuit,26.0325,50.5106,sakhir,Bahrain,2026-11-08T09:46:26.697Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/circuits.csv,2025-01


#### Step 5 - Filter out rows where circuit_id is null (business key validation)

In [0]:
# circuits_valid_df = circuits_renamed_df.filter(
#     "circuit_id IS NOT NULL"
# )

In [0]:
circuits_valid_df = circuits_renamed_df.filter(
    F.col("circuit_id").isNotNull()
)

In [0]:
# display(circuits_valid_df)

#### Step 6 - Remove duplicate records

In [0]:
# circuits_distinct_df = circuits_valid_df.distinct()

In [0]:
circuits_distinct_df = circuits_valid_df.dropDuplicates(["circuit_id"])

In [0]:
# display(circuits_distinct_df)

#### Step 7 - Transform values of columns `circuit_name` and `locality` to Title Case

In [0]:
circuits_final_df = (
    circuits_distinct_df
        .withColumn('circuit_name', F.initcap(F.col("circuit_name")))
        .withColumn('locality', F.initcap(F.col("locality")))
)

In [0]:
# display(circuits_final_df)

#### Step 8 - Write the transformed data to silver `circuits` table

In [0]:

# -- CREATE SCHEMA IF NOT EXISTS formula1.silver
# --    MANAGED LOCATION 'abfss://formula1@databrickscourseextdi123.dfs.core.windows.net/silver';

In [0]:
circuits_final_df =(
    circuits_final_df
        .withColumn("created_timestamp", F.current_timestamp())
        .withColumn("updated_timestamp", F.current_timestamp())
)

In [0]:
# from delta.tables import DeltaTable
# if not spark.catalog.tableExists(silver_table):
#     (
#         circuits_final_df
#             .write
#             .format("delta")
#             .mode("overwrite")
#             .saveAsTable(silver_table)
#     )
# else:
#     delta_table = DeltaTable.forName(spark, silver_table)
#     (
#     delta_table.alias("t")
#     .merge(
#         circuits_final_df.alias("s"),
#         "t.circuit_id = s.circuit_id"
#     )
#     .whenMatchedUpdate(
#            condition="s.batch_id >= t.batch_id",
#         set ={
#             "circuit_name": "s.circuit_name",
#             "latitude": "s.latitude",
#             "longitude": "s.longitude",
#             "country": "s.country",
#             "ingestion_timestamp": "s.ingesion_timestamp",
#             "source_file": "s.source_file",
#             "batch_id": "s.batch_id",
#             "updated_timestamp": "s.updated_timestamp"
#         }
#     )
#     .whenNotMatchedInsertAll()
#     .execute()
#     )
# # display(spark.table(silver_table))
# #
    


In [0]:
%sql
--DROP TABLE formula1_incr.silver.circuits

In [0]:
write_to_silver(
    input_df=circuits_final_df,
    target_table=silver_table,
    merge_condition="t.circuit_id = s.circuit_id",
    columns_to_update=[
        "circuit_name",
        "latitude",
        "longitude",
        "locality",
        "country",
        "ingestion_timestamp",
        "source_file",
        "batch_id"
    ]
)

In [0]:
# (
#     circuits_final_df
#         .write
#         .format("delta")
#         .mode("overwrite")
#         .saveAsTable(silver_table)
# )

In [0]:
 display(spark.table(silver_table))

circuit_id,circuit_name,latitude,longitude,locality,country,ingestion_timestamp,source_file,batch_id,created_timestamp,updated_timestamp
bahrain,Bahrain International Circuit,26.0325,50.5106,Sakhir,Bahrain,2026-11-08T09:46:26.697Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/circuits.csv,2025-01,2026-11-08T14:05:39.741Z,2026-11-08T14:05:57.248Z
detroit,Detroit Street Circuit,42.3298,-83.0401,Detroit,USA,2026-11-08T09:46:26.697Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/circuits.csv,2025-01,2026-11-08T14:05:39.741Z,2026-11-08T14:05:57.248Z
mosport,Mosport International Raceway,44.0481,-78.6756,Ontario,Canada,2026-11-08T09:46:26.697Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/circuits.csv,2025-01,2026-11-08T14:05:39.741Z,2026-11-08T14:05:57.248Z
red_bull_ring,Red Bull Ring,47.2197,14.7647,Spielberg,Austria,2026-11-08T09:46:26.697Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/circuits.csv,2025-01,2026-11-08T14:05:39.741Z,2026-11-08T14:05:57.248Z
sebring,Sebring International Raceway,27.4547,-81.3483,Florida,USA,2026-11-08T09:46:26.697Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/circuits.csv,2025-01,2026-11-08T14:05:39.741Z,2026-11-08T14:05:57.248Z
adelaide,Adelaide Street Circuit,-34.9272,138.617,Adelaide,Australia,2026-11-08T09:46:26.697Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/circuits.csv,2025-01,2026-11-08T14:05:39.741Z,2026-11-08T14:05:57.248Z
buddh,Buddh International Circuit,28.3487,77.5331,Uttar Pradesh,India,2026-11-08T09:46:26.697Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/circuits.csv,2025-01,2026-11-08T14:05:39.741Z,2026-11-08T14:05:57.248Z
monaco,Circuit De Monaco,43.7347,7.42056,Monte Carlo,Monaco,2026-11-08T09:46:26.697Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/circuits.csv,2025-01,2026-11-08T14:05:39.741Z,2026-11-08T14:05:57.248Z
pedralbes,Circuit De Pedralbes,41.3903,2.11667,Barcelona,Spain,2026-11-08T09:46:26.697Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/circuits.csv,2025-01,2026-11-08T14:05:39.741Z,2026-11-08T14:05:57.248Z
tremblant,Circuit Mont-tremblant,46.1877,-74.6099,Quebec,Canada,2026-11-08T09:46:26.697Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/circuits.csv,2025-01,2026-11-08T14:05:39.741Z,2026-11-08T14:05:57.248Z
